In [46]:
from src.improved_model import  BinarizingCNN
from src.training import test_dataset, train_dataset
from src.utils import device
from settings import settings
from torch.utils.data import DataLoader
import torch

model = BinarizingCNN()
model.to(device)
model.load_state_dict(torch.load(settings.models_path / "convnet_v2.pth"))
model.eval_mode()
model.binarize_weights()
train_dataloader = DataLoader(train_dataset, batch_size=int(5e4), shuffle=True)
train_data, _ = next(iter(train_dataloader))
train_data = train_data.to(device)

C:\Users\frrit\AppData\Local\Temp\ipykernel_22680\3071379615.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(settings.models_path / "co

In [6]:
train_data, _  = next(iter(train_dataloader))
train_data = train_data.to(device)

x = model.second_layer(model.float_to_binary_layer(train_data))



In [6]:
cnn_model = BinarizingCNN()
cnn_model.load_state_dict(torch.load(settings.models_path / 'convnet_v2.pth'))
cnn_model.to(device)
cnn_model.eval_mode()

old_bias = cnn_model.layer3.bias.clone()
old_output = cnn_model(train_data)

cnn_model.third_layer = lambda x: torch.where(cnn_model.layer3(x) >= 0, torch.tensor(1), torch.tensor(-1))
new_bias = torch.floor(old_bias.clone())
cnn_model.layer3.bias = torch.nn.Parameter(new_bias)
new_output = cnn_model(train_data)

assert  torch.all(old_output == new_output)

C:\Users\frrit\AppData\Local\Temp\ipykernel_35956\606248072.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn_model.load_state_dict(torch.load(settings.models_path / '

In [7]:
new_output.dtype

torch.int64

In [8]:
new_output[old_output != new_output]


tensor([], device='cuda:0', dtype=torch.int64)

In [47]:
from torch import nn, Tensor

y_before_pruning = model(train_data)


x = model.second_layer(model.float_to_binary_layer(train_data))

def prune_model_layers(first_layer: BinarizingLinear, second_layer: BinarizingLinear, output_first_layer: Tensor) -> None:
    # Check if all values in each column are the same
    same_values_per_column = (output_first_layer == output_first_layer[0]).all(dim=0)
    # Get the indices of columns where all values are the same
    columns_to_remove = torch.nonzero(same_values_per_column, as_tuple=True)[0]
    columns_to_keep = [i for i in range(output_first_layer.size(1)) if i not in columns_to_remove]
    new_bias = output_first_layer[0, columns_to_remove] @ model.layer3.weight[:, columns_to_remove].t()
    model.layer2.weight = nn.Parameter(model.layer2.weight[columns_to_keep, :])
    model.layer2.bias =  nn.Parameter(model.layer2.bias[columns_to_keep])
    model.layer3.weight = nn.Parameter(model.layer3.weight[:, columns_to_keep])
    model.layer3.bias =  nn.Parameter(new_bias + model.layer3.bias)


y_after_pruning = model(train_data)

assert torch.all(y_before_pruning == y_after_pruning)

In [48]:
print(model.layer3


BinarizingLinear(in_features=77, out_features=10, bias=True)
